In [0]:
dbutils.widgets.text("table_name", "", "Bronze Table Name")

table_name = dbutils.widgets.get("table_name").strip()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

if not table_name: raise ValueError("table_name parameter is required.")

print("=" * 80)
print(f"Starting Silver Transformation")
print(f"Table : {table_name}")
print("=" * 80)


# ============================================================
# 2. SOURCE / TARGET
# ============================================================

SOURCE_CATALOG = "f1_catalog"
SOURCE_SCHEMA = "bronze"

TARGET_CATALOG = "f1_catalog"
TARGET_SCHEMA = "silver"

source_table = f"{SOURCE_CATALOG}.{SOURCE_SCHEMA}.{table_name}"
target_table = f"{TARGET_CATALOG}.{TARGET_SCHEMA}.{table_name}"

print(f"Source : {source_table}")
print(f"Target : {target_table}")


# ============================================================
# 3. TABLE CONFIGURATION
# ============================================================

table_config = {

    "drivers": {
        "primary_keys": ["driver_id"],

        "rename": {
            "driverId": "driver_id",
            "driverRef": "driver_ref",
            "forename": "first_name",
            "surname": "last_name"
        },

        "cast": {
            "driver_id": "integer",
            "number": "integer"
        },

        "dates": {
            "dob": "date_of_birth"
        }
    },


    "constructors": {
        "primary_keys": ["constructor_id"],

        "rename": {
            "constructorId": "constructor_id",
            "constructorRef": "constructor_ref",
            "name": "constructor_name"
        },

        "cast": {
            "constructor_id": "integer"
        }
    },


    "circuits": {
        "primary_keys": ["circuit_id"],

        "rename": {
            "circuitId": "circuit_id",
            "circuitRef": "circuit_ref",
            "name": "circuit_name"
        },

        "cast": {
            "circuit_id": "integer",
            "lat": "double",
            "lng": "double",
            "alt": "integer"
        }
    },


    "races": {
        "primary_keys": ["race_id"],

        "rename": {
            "raceId": "race_id",
            "year": "season",
            "round": "round_number",
            "circuitId": "circuit_id"
        },

        "cast": {
            "race_id": "integer",
            "season": "integer",
            "round_number": "integer",
            "circuit_id": "integer"
        },

        "dates": {
            "date": "race_date"
        }
    },


    "results": {
        "primary_keys": ["result_id"],

        "rename": {
            "resultId": "result_id",
            "raceId": "race_id",
            "driverId": "driver_id",
            "constructorId": "constructor_id",
            "number": "car_number",
            "position": "finish_position",
            "positionText": "position_text",
            "positionOrder": "position_order",
            "fastestLap": "fastest_lap",
            "fastestLapTime": "fastest_lap_time",
            "fastestLapSpeed": "fastest_lap_speed"
        },

        "cast": {
            "result_id": "integer",
            "race_id": "integer",
            "driver_id": "integer",
            "constructor_id": "integer",
            "car_number": "integer",
            "grid": "integer",
            "finish_position": "integer",
            "position_order": "integer",
            "points": "double",
            "laps": "integer",
            "fastest_lap": "integer",
            "fastest_lap_speed": "double"
        }
    },


    "qualifying": {
        "primary_keys": ["qualify_id"],

        "rename": {
            "qualifyId": "qualify_id",
            "raceId": "race_id",
            "driverId": "driver_id",
            "constructorId": "constructor_id",
            "number": "car_number",
            "position": "qualifying_position"
        },

        "cast": {
            "qualify_id": "integer",
            "race_id": "integer",
            "driver_id": "integer",
            "constructor_id": "integer",
            "car_number": "integer",
            "qualifying_position": "integer"
        }
    },


    "pit_stops": {
        "primary_keys": [
            "race_id",
            "driver_id",
            "stop"
        ],

        "rename": {
            "raceId": "race_id",
            "driverId": "driver_id",
            "lap": "lap_number"
        },

        "cast": {
            "race_id": "integer",
            "driver_id": "integer",
            "stop": "integer",
            "lap_number": "integer",
            "duration": "double"
        }
    },


    "lap_times": {
        "primary_keys": [
            "race_id",
            "driver_id",
            "lap_number"
        ],

        "rename": {
            "raceId": "race_id",
            "driverId": "driver_id",
            "lap": "lap_number"
        },

        "cast": {
            "race_id": "integer",
            "driver_id": "integer",
            "lap_number": "integer",
            "position": "integer",
            "milliseconds": "long"
        }
    },


    "driver_standings": {
        "primary_keys": ["driver_standings_id"],

        "rename": {
            "driverStandingsId": "driver_standings_id",
            "raceId": "race_id",
            "driverId": "driver_id",
            "position": "standing_position"
        },

        "cast": {
            "driver_standings_id": "integer",
            "race_id": "integer",
            "driver_id": "integer",
            "standing_position": "integer",
            "points": "double",
            "wins": "integer"
        }
    },


    "constructor_standings": {
        "primary_keys": ["constructor_standings_id"],

        "rename": {
            "constructorStandingsId": "constructor_standings_id",
            "raceId": "race_id",
            "constructorId": "constructor_id",
            "position": "standing_position"
        },

        "cast": {
            "constructor_standings_id": "integer",
            "race_id": "integer",
            "constructor_id": "integer",
            "standing_position": "integer",
            "points": "double",
            "wins": "integer"
        }
    },


    "seasons": {
        "primary_keys": ["year"],

        "rename": {},

        "cast": {
            "year": "integer"
        }
    },


    "status": {
        "primary_keys": ["status_id"],

        "rename": {
            "statusId": "status_id"
        },

        "cast": {
            "status_id": "integer"
        }
    }
}


# ============================================================
# 4. CHECK TABLE CONFIGURATION
# ============================================================

if table_name not in table_config:

    raise ValueError(
        f"No Silver configuration found for table: {table_name}"
    )
    

config = table_config[table_name]

# ============================================================
# 4A. READ LAST WATERMARK
# ============================================================

print("\nReading last watermark...")

watermark_table = "f1_catalog.gold.pipeline_watermark"

wm_df = (
    spark.table(watermark_table)
    .filter(
        F.col("table_name") == table_name
    )
)

if wm_df.count() == 0:

    last_watermark = "1900-01-01 00:00:00"

    print(
        f"No previous watermark found for {table_name}. "
        f"First load will process all records."
    )

else:

    last_watermark = (wm_df.select("processed_timestamp").first()["processed_timestamp"])

    print(f"Previous watermark for {table_name}: " f"{last_watermark}")


# ============================================================
# 5. READ FROM BRONZE
# ============================================================

print("\nReading Bronze table...")

df = spark.table(source_table)

source_count = df.count()

print(f"Bronze records: {source_count}")

if source_count == 0:
    raise ValueError(
        f"Bronze table {source_table} contains zero records."
    )


# ============================================================
# 6. DISPLAY SOURCE SCHEMA
# ============================================================

print("\nBronze Schema:")

df.printSchema()


# ============================================================
# 7. RENAME COLUMNS
# ============================================================

print("\nApplying column renaming...")

for old_name, new_name in config.get("rename", {}).items():

    if old_name in df.columns:

        df = df.withColumnRenamed(
            old_name,
            new_name
        )

        print(f"Renamed: {old_name} -> {new_name}")


# ============================================================
# 8. TRIM STRING COLUMNS
# ============================================================

print("\nTrimming string columns...")

for field in df.schema.fields:

    if isinstance(field.dataType, StringType):

        df = df.withColumn(
            field.name,
            F.trim(F.col(field.name))
        )


# ============================================================
# 9. CAST DATA TYPES
# ============================================================

print("\nApplying data type conversions...")

for column_name, data_type in config.get("cast", {}).items():

    if column_name in df.columns:

        df = df.withColumn(
            column_name,
            F.col(column_name).try_cast(data_type)
        )

        print(
            f"Cast: {column_name} -> {data_type}"
        )


# ============================================================
# 10. DATE CONVERSION
# ============================================================

print("\nApplying date transformations...")

for source_column, target_column in config.get("dates", {}).items():

    if source_column in df.columns:

        df = df.withColumn(
            target_column,
            F.to_date(F.col(source_column))
        )

        if source_column != target_column:

            df = df.drop(source_column)

        print(
            f"Date conversion: "
            f"{source_column} -> {target_column}"
        )


# ============================================================
# 11. REMOVE DUPLICATES
# ============================================================

primary_keys = config.get("primary_keys", [])

existing_keys = [
    key for key in primary_keys
    if key in df.columns
]

if existing_keys:

    before_count = df.count()

    df = df.dropDuplicates(existing_keys)

    after_count = df.count()

    duplicate_count = before_count - after_count

    print(
        f"\nDuplicate records removed: "
        f"{duplicate_count}"
    )

else:

    print(
        "\nWARNING: No valid primary key columns found."
    )


# ============================================================
# 12. DATA QUALITY VALIDATION
# ============================================================

print("\nRunning Data Quality checks...")

dq_failed = False

for key in existing_keys:

    null_count = (
        df
        .filter(F.col(key).isNull())
        .count()
    )

    print(
        f"{key} NULL count: {null_count}"
    )

    if null_count > 0:

        dq_failed = True


if dq_failed:

    raise ValueError(
        f"Data Quality validation failed for "
        f"table: {table_name}"
    )


# ============================================================
# 13. ADD SILVER TECHNICAL COLUMNS
# ============================================================

print("\nAdding Silver metadata...")

df = (
    df
    .withColumn(
        "_silver_processed_timestamp",
        F.current_timestamp()
    )
    .withColumn(
        "_source_table",
        F.lit(source_table)
    )
)


# ============================================================
# 14. FINAL RECORD COUNT
# ============================================================

silver_count = df.count()

print(
    f"\nSilver records: {silver_count}"
)


# ============================================================
# 15. WRITE TO SILVER
# ============================================================

print("\nWriting to Silver...")

(
    df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)


# ============================================================
# 16. VALIDATION AFTER WRITE
# ============================================================

target_count = spark.table(
    target_table
).count()

print(
    f"Silver target records: {target_count}"
)


# ============================================================
# 17. COMPLETION MESSAGE
# ============================================================

print("\n" + "=" * 80)
print("SILVER TRANSFORMATION SUCCESSFUL")
print("=" * 80)

print(f"Table       : {table_name}")
print(f"Source      : {source_table}")
print(f"Target      : {target_table}")
print(f"Source rows : {source_count}")
print(f"Target rows : {target_count}")

print("=" * 80)


# ============================================================
# 18. RETURN RESULT TO PARENT
# ============================================================

dbutils.notebook.exit(
    f"SUCCESS | {table_name} | "
    f"Bronze={source_count} | Silver={target_count}"
)

In [0]:
%sql
SELECT * FROM f1_catalog.gold.data_quality_audit